# Concrete Strength Prediction

**Tabular Regression Project** — Predict concrete compressive strength.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: UCI Concrete (1,030 rows × 9 columns)
Target: `Concrete compressive strength`

## 2 · Project Overview

We predict the **compressive strength of concrete** based on its mix proportions (cement, water, aggregates, admixtures) and curing age. This is a classic materials science regression problem.

The dataset has 1,030 samples — a mid-small dataset where feature engineering and model selection both matter.

## 3 · Learning Objectives

1. Download datasets from Kaggle using kagglehub.
2. Understand concrete mix design parameters.
3. Apply domain-informed feature engineering.
4. Compare models on a materials science dataset.
5. Evaluate with RMSE, MAE, R².

## 4 · Problem Statement

Given the **mix proportions** of a concrete batch (cement, slag, fly ash, water, superplasticizer, coarse/fine aggregate) and **curing age**, predict the **compressive strength** in MPa.

## 5 · Why This Project Matters

- **Concrete strength prediction** saves time and money in construction.
- Testing concrete takes 28 days — ML can predict strength earlier.
- The dataset illustrates a clean regression problem with domain knowledge available.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 1,030 |
| **Features** | Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate, Age |
| **Target** | Concrete compressive strength (MPa) |
| **Source** | Kaggle: elikplim/concrete-compressive-strength-data-set |

## 7 · Dataset Source and License Notes

- **Source**: UCI ML Repository (Yeh, 1998), mirrored on Kaggle.
- **License**: Public domain for research.
- **Citation**: I-Cheng Yeh, 'Modeling of strength of high-performance concrete using artificial neural networks', Cement and Concrete Research, 1998.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub

# Download the dataset
try:
    path = kagglehub.dataset_download('elikplim/concrete-compressive-strength-data-set')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download failed: {e}')
    path = SAVE_DIR

# Find CSV file
import glob
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
if not csv_files:
    csv_files = glob.glob(os.path.join(path, '**', '*.xls*'), recursive=True)

assert csv_files, f'No data files found in {path}'
data_file = sorted(csv_files, key=os.path.getsize, reverse=True)[0]
print(f'Using: {data_file}')

if data_file.endswith('.csv'):
    df = pd.read_csv(data_file)
else:
    df = pd.read_excel(data_file)

print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
# Standardize column names
df.columns = [c.strip() for c in df.columns]

# Find target column
target_candidates = [c for c in df.columns if 'strength' in c.lower() or 'compressive' in c.lower()]
TARGET = target_candidates[0] if target_candidates else df.columns[-1]
print(f'Target column: {TARGET}')

print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

feature_cols = [c for c in df.columns if c != TARGET]
for ax, col in zip(axes.flat, feature_cols[:4]):
    ax.scatter(df[col], df[TARGET], alpha=0.4, s=10)
    ax.set_xlabel(col[:20]); ax.set_ylabel('Strength')
    ax.set_title(f'{col[:20]} vs Strength')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
df[TARGET].hist(bins=30, ax=ax, edgecolor='black')
ax.set_title('Concrete Strength Distribution')
ax.set_xlabel('Compressive Strength (MPa)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'target_analysis.png'), dpi=100)
plt.show()
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are numeric (material quantities and age). No encoding needed.

## 17 · Feature Engineering

We add water-cement ratio and total binder content — important in concrete science.

In [ ]:
cement_col = [c for c in X_train.columns if 'cement' in c.lower()][0]
water_col = [c for c in X_train.columns if 'water' in c.lower()][0]
age_col = [c for c in X_train.columns if 'age' in c.lower()][0]

for df_part in [X_train, X_test]:
    df_part['water_cement_ratio'] = df_part[water_col] / (df_part[cement_col] + 0.1)
    df_part['log_age'] = np.log1p(df_part[age_col])
    # Total binder
    slag_cols = [c for c in df_part.columns if 'slag' in c.lower()]
    ash_cols = [c for c in df_part.columns if 'ash' in c.lower() or 'fly' in c.lower()]
    binder = df_part[cement_col].copy()
    for sc in slag_cols: binder = binder + df_part[sc]
    for ac in ash_cols: binder = binder + df_part[ac]
    df_part['total_binder'] = binder

print(f'Features after engineering: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Cement content** and **Age** are the strongest predictors.
- **Water-cement ratio** (engineered) is known to be the most important factor in concrete science — our models confirm this.
- Superplasticizer allows lower water content while maintaining workability.
- Predictions can help construction teams optimize mix designs for target strength.

## 26 · Limitations

- Only 1,030 samples.
- No information about mixing/curing conditions.
- Real concrete has many more variables (ambient temperature, humidity).
- Dataset may not represent modern concrete formulations.

## 27 · How to Improve This Project

1. Add curing condition features.
2. Try polynomial features for water-cement interaction.
3. Collect more data from different sources.
4. Hyperparameter tune the top model.

## 28 · Production Considerations

- Need lab-validated predictions before real use.
- Different aggregate sources change behavior.
- Regulatory requirements for concrete testing.
- Model should augment, not replace, standard testing.

## 29 · Common Mistakes

1. Not engineering water-cement ratio — a known key parameter.
2. Not log-transforming Age (strength gain slows over time).
3. Ignoring total binder content.
4. Over-fitting on 1,030 rows with too many features.

## 30 · Mini Challenge / Exercises

1. Plot strength vs age curve at different cement levels.
2. Try adding interaction features between all material pairs.
3. Use cross-validation instead of single split.
4. Research ACI equations and compare with ML predictions.

## 31 · Final Summary / Key Takeaways

- Domain knowledge (water-cement ratio, total binder) improves predictions.
- Tree-based models capture non-linear material interactions effectively.
- Age and cement are the dominant features.
- This dataset is a clean example of materials informatics.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')